# GRU vs XGBoost: ensemble comparison (val 2022)

What is tested: whether the per-generator GRU (v2, micro 0.9579 on val
2022) carries signal complementary to the per-generator XGBoost baseline
(0.9802), by comparing mean-probability and mean-rank ensembles against
XGBoost alone.

Protocol: XGBoost retrained exactly as `notebooks/xgboost_baseline.ipynb`
(train 2015-2021, train_tail stop, full 2022 val, seed 0); GRU val-2022
predictions loaded from `../gru_val_preds.npz` (saved by
`gru_sequence.ipynb` v2). No LOYO, no retraining of the GRU.

This notebook must run in a fresh kernel and never imports torch:
torch's MPS runtime and xgboost's OpenMP threads crashed the kernel twice
when sharing a process on this Mac. Sanity checks: the XGBoost cell must
print micro 0.9802 and the loaded GRU predictions micro 0.9579 — if either
differs, stop and find why before reading the ensemble numbers.

Decision rule: if neither ensemble beats 0.9802 beyond noise, the GRU is
dropped and the submission uses XGBoost only.

In [ ]:
import gc

import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import roc_auc_score

HOURS = 168

In [ ]:
DATA = "../data/kernel"
EXT = "../data/extended"

price = pd.read_csv(f"{DATA}/Historical_day_ahead_price_2015_2025.csv")
inflow = pd.read_csv(f"{DATA}/Historical_inflow_1958_2025.csv")
volume = pd.read_csv(f"{DATA}/Historical_volume_2015_2024.csv")
water = pd.read_csv(f"{DATA}/Synthetic_water_value_2015_2024.csv")
target = pd.read_csv(f"{DATA}/Unit_commitment_decisions.csv")
min_flow = pd.read_csv(f"{EXT}/Constraint_min_flow.csv")
min_volume = pd.read_csv(f"{EXT}/Constraint_min_volume.csv")
topology = yaml.safe_load(open(f"{EXT}/Tokke_Vinje_topology.yaml"))


def indexed(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, format="mixed")
    return df.sort_values("date").set_index("date")


price_h = indexed(price)["Day-ahead price (EUR/MWh)"].resample("1h").mean()
inf_wide = indexed(inflow)
vol_wide = indexed(volume)
wv_wide = indexed(water)
mf_wide = indexed(min_flow)
mv_wide = indexed(min_volume)

RESERVOIRS = list(vol_wide.columns)
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}

target["starttime"] = pd.to_datetime(target["starttime"], utc=True, format="mixed")
target = target.sort_values("starttime").reset_index(drop=True)

GENERATORS = sorted(
    {
        c.removeprefix("result_committed_").rsplit("_t", 1)[0]
        for c in target.columns
        if c.startswith("result_committed")
    }
)

connections = topology["connections"]


def supplying_reservoirs(name, kind):
    if kind == "reservoir":
        return {name}
    out = set()
    for e in connections:
        if (
            e["to"] == name
            and e["to_type"] == kind
            and e["from_type"] in {"reservoir", "tunnel"}
        ):
            out |= supplying_reservoirs(e["from"], e["from_type"])
    return out


FEEDERS = {}
for gen in GENERATORS:
    plant = next(
        e["to"]
        for e in connections
        if e["from"] == gen
        and e["from_type"] == "generator"
        and e["to_type"] == "plant"
    )
    FEEDERS[gen] = sorted(supplying_reservoirs(plant, "plant"))

print("cases:", len(target), " generators:", len(GENERATORS))

cases: 2922  generators: 14


## Feature engineering and split

What is tested: the unchanged 193-column feature set and baseline split,
copied verbatim from `notebooks/xgboost_baseline.ipynb`.

In [3]:
def lookup(frame, timestamps, columns=None):
    f = frame if columns is None else frame[columns]
    return f.reindex(pd.DatetimeIndex(timestamps)).to_numpy(dtype=np.float32)


def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    price = lookup(price_h, ts).reshape(n, HOURS)
    initial = lookup(vol_wide, starts, RESERVOIRS)
    terminal = lookup(wv_wide, starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflow = lookup(inf_wide, ts).reshape(n, HOURS, -1)
    minflow = lookup(mf_wide, ts).reshape(n, HOURS, -1)
    minvol = lookup(mv_wide, ts).reshape(n, HOURS, -1)

    F = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        F[name] = v

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", ts.hour.to_numpy())
    add("day_of_week", ts.dayofweek.to_numpy())
    add("month", ts.month.to_numpy())
    add("doy_sin", np.sin(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))
    add("doy_cos", np.cos(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))

    add("price", price)
    for stat, v in (
        ("mean", price.mean(1)),
        ("std", price.std(1)),
        ("min", price.min(1)),
        ("max", price.max(1)),
    ):
        add(f"price_week_{stat}", v)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, v in (
        ("mean", daily.mean(2)),
        ("min", daily.min(2)),
        ("max", daily.max(2)),
    ):
        add(f"price_day_{stat}", np.repeat(v, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add(
        "price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1)
    )
    for off in (-24, -6, -1, 1, 6, 24):
        add(
            f"price_offset_{off:+d}",
            price[:, np.clip(np.arange(HOURS) + off, 0, HOURS - 1)],
        )

    for j, res in enumerate(RESERVOIRS):
        add(f"init_vol__{res}", initial[:, j])
        add(f"init_frac__{res}", initial[:, j] / CAPACITY[res])
        add(f"term_val__{res}", terminal[:, j])
        add(f"price_minus_val__{res}", price - terminal[:, j, None])

    for j, name in enumerate(inf_wide.columns):
        flow = inflow[:, :, j]
        cum = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cum_Mm3__{name}", cum)
        add(f"inflow_rem_Mm3__{name}", cum[:, -1:] - cum)

    for j, name in enumerate(mf_wide.columns):
        add(f"minflow__{name}", minflow[:, :, j])
        add(f"minflow_week_max__{name}", minflow[:, :, j].max(1))
    for j, name in enumerate(mv_wide.columns):
        add(f"minvol__{name}", minvol[:, :, j])
        add(f"minvol_week_max__{name}", minvol[:, :, j].max(1))
        add(
            f"init_margin_to_minvol__{name}",
            initial[:, RESERVOIRS.index(name), None] - minvol[:, :, j],
        )

    return pd.DataFrame(F, dtype=np.float32)


def generator_features(base, gen):
    X = base.copy()
    feeders = FEEDERS[gen]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_init = sum(base[f"init_vol__{r}"] for r in feeders)
    value = sum(base[f"term_val__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cum = sum(base[f"inflow_cum_Mm3__{r}"] for r in feeders)
    rem = sum(base[f"inflow_rem_Mm3__{r}"] for r in feeders)

    X["local_init_frac"] = total_init / capacity
    X["local_term_val"] = value
    X["local_price_minus_val"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cum_frac"] = cum / capacity
    X["local_rem_frac"] = rem / capacity
    X["local_available_frac"] = (total_init + cum + rem) / capacity
    return X


def labels_for(cases, gen):
    cols = [f"result_committed_{gen}_t{t}" for t in range(HOURS)]
    return cases[cols].to_numpy(dtype=np.uint8).reshape(-1)


def auc_or_nan(y, p):
    return roc_auc_score(y, p) if np.unique(y).size == 2 else float("nan")


split = pd.Timestamp("2022-01-01", tz="UTC")
train_cases = target[target["starttime"] < split].reset_index(drop=True)
val_cases = target[target["starttime"] >= split].reset_index(drop=True)
stop_cases = train_cases[
    train_cases["starttime"] >= split - pd.Timedelta(days=90)
].reset_index(drop=True)
print(
    "train cases:",
    len(train_cases),
    " stop cases:",
    len(stop_cases),
    " val cases:",
    len(val_cases),
)

Xtr_base = feature_matrix(train_cases)
Xst_base = feature_matrix(stop_cases)
Xva_base = feature_matrix(val_cases)
print("feature matrix:", Xtr_base.shape)

train cases: 2557  stop cases: 90  val cases: 365
feature matrix: (429576, 186)


## XGBoost retrain

What is tested: the 14 baseline XGBoost models, identical to
`notebooks/xgboost_baseline.ipynb` (seed 0, train_tail stop). Sanity:
micro AUC must print 0.9802. Per-generator copies and fitted models are
deleted after use to keep memory low (the historical crashes in this
comparison happened in the XGBoost cell).

In [4]:
xgb_preds = {}
for gen in GENERATORS:
    Xtr = generator_features(Xtr_base, gen)
    Xst = generator_features(Xst_base, gen)
    Xva = generator_features(Xva_base, gen)
    ytr = labels_for(train_cases, gen)
    yst = labels_for(stop_cases, gen)
    yva = labels_for(val_cases, gen)

    if np.unique(ytr).size == 1:
        xgb_preds[gen] = np.full(len(yva), float(ytr[0]), dtype=np.float32)
    else:
        clf = xgb.XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=7,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            eval_metric="auc",
            early_stopping_rounds=25,
            random_state=0,
        )
        clf.fit(Xtr, ytr, eval_set=[(Xst, yst)], verbose=False)
        xgb_preds[gen] = clf.predict_proba(Xva)[:, 1].astype(np.float32)
        del clf
    del Xtr, Xst, Xva
    gc.collect()
    print(f"{gen:16s} AUC {auc_or_nan(yva, xgb_preds[gen]):.4f}")

y_all = np.concatenate([labels_for(val_cases, g) for g in GENERATORS])
p_xgb = np.concatenate([xgb_preds[g] for g in GENERATORS])
xgb_micro = roc_auc_score(y_all, p_xgb)
print("XGB micro AUC:", round(xgb_micro, 4))

del Xtr_base, Xst_base
gc.collect()

Byrte_G1         AUC 0.9951
Haukeli_G1       AUC 0.9352
Hogga_G1         AUC 0.9722
Kjela_G1         AUC 0.9789
Lio_G1           AUC 0.9225
Songa_G1         AUC 0.9966
Tokke_G1         AUC 0.9928
Tokke_G2         AUC 0.9839
Tokke_G3         AUC 0.9886
Tokke_G4         AUC 0.9810
Vesle Kjela_G1   AUC 0.9815
Vinje_G1         AUC 0.9835
Vinje_G2         AUC 0.9765
Vinje_G3         AUC 0.9701
XGB micro AUC: 0.9802


0

## GRU predictions and the ensembles

What is tested: GRU val-2022 predictions loaded from `../gru_val_preds.npz`
(sanity: micro 0.9579), then two ensembles against XGBoost alone — mean
probability, and mean of within-generator percentile ranks (the pooled
micro AUC mixes generators, so ranks are computed per generator). Printed:
all four micro AUCs and a per-generator table (XGB, GRU, both ensembles)
sorted by GRU AUC.

In [5]:
z = np.load("../gru_val_preds.npz")
gru_preds = {g: z[g].reshape(-1).astype(np.float32) for g in GENERATORS}
p_gru = np.concatenate([gru_preds[g] for g in GENERATORS])
gru_micro = roc_auc_score(y_all, p_gru)
print("GRU micro AUC (loaded):", round(gru_micro, 4))


def rank(p):
    return pd.Series(p).rank(pct=True).to_numpy(dtype=np.float32)


p_mean = (p_gru + p_xgb) / 2
p_rank = np.concatenate(
    [(rank(gru_preds[g]) + rank(xgb_preds[g])) / 2 for g in GENERATORS]
)
mean_micro = roc_auc_score(y_all, p_mean)
rank_micro = roc_auc_score(y_all, p_rank)

print("XGB           :", round(xgb_micro, 4))
print("GRU           :", round(gru_micro, 4))
print("mean ensemble :", round(mean_micro, 4))
print("rank ensemble :", round(rank_micro, 4))

rows = []
for gen in GENERATORS:
    yv = labels_for(val_cases, gen)
    rows.append(
        {
            "generator": gen,
            "xgb": auc_or_nan(yv, xgb_preds[gen]),
            "gru": auc_or_nan(yv, gru_preds[gen]),
            "mean": auc_or_nan(yv, (gru_preds[gen] + xgb_preds[gen]) / 2),
            "rank": auc_or_nan(yv, (rank(gru_preds[gen]) + rank(xgb_preds[gen])) / 2),
        }
    )
pd.DataFrame(rows).set_index("generator").round(4).sort_values("gru")

GRU micro AUC (loaded): 0.9579
XGB           : 0.9802
GRU           : 0.9579
mean ensemble : 0.9809
rank ensemble : 0.9556


,xgb,gru,mean,rank
generator,,,,
Haukeli_G1,0.9352,0.8850,0.9463,0.9464
Lio_G1,0.9225,0.9273,0.9570,0.9638
Vinje_G3,0.9701,0.9566,0.9687,0.9682
Kjela_G1,0.9789,0.9575,0.9791,0.9846
Hogga_G1,0.9722,0.9611,0.9729,0.9743
Byrte_G1,0.9951,0.9743,0.9917,0.9880
Vinje_G2,0.9765,0.9752,0.9795,0.9805
Vinje_G1,0.9835,0.9794,0.9855,0.9854
Tokke_G3,0.9886,0.9801,0.9891,0.9898
